# Screen accounts for simulated future reports

Which accounts will receive a simulated suspicious-activity report in the next 30 days? Compare Kumo Engine with logistic regression on historical holdout accounts, then score newly added accounts from the same synthetic process.

**Kumo Engine · Databricks GPU notebook · Synthetic data**

Follow the [money-laundering prediction workflow](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/workflows-and-solutions/fraud/money-laundering-detection). A report score prioritizes review; it does not establish money-mule activity or wrongdoing.


## Install Kumo Engine

Use Databricks Runtime 17.3 LTS ML GPU on a single-node cluster with dedicated access, or Serverless GPU environment version 6.
Install the latest package below, following the [Databricks notebook setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/databricks-notebook).
The synthetic example uses driver-local files and needs no warehouse credentials or Unity Catalog volume.


In [ ]:
%pip install "kumo-engine[databricks]"


Restart Python to load the installed packages before defining any variables.


In [ ]:
dbutils.library.restartPython()


<br>

## Configure the example

Use the defaults to generate a small account network and a short demonstration run.
Each run creates its own directory under `/tmp/kumoai-local` for data, scratch files, and saved predictions.


In [ ]:
# Synthetic data and the frozen observation date.
SEED = 42
N_ACCOUNTS = 240
N_TRANSFERS = 2400
N_NEW_ACCOUNTS = 40
N_NEW_TRANSFERS = 400
CUTOFF = "2026-10-01"
HISTORY_DAYS = 180
HORIZON_DAYS = 30
REPORT_BASE_RATE = 0.04
REPORT_SIGNAL = 0.65
SPLIT_FRACTIONS = (0.60, 0.15, 0.10, 0.15)

# Scoring and output choices.
THRESHOLD = 0.5
OUTPUT_PARENT = "/tmp/kumoai-local"  # Temporary driver-local files.
CLEANUP = False       # Keep predictions and saved model files after the run.


Import the tools and create this run's data and scratch directories.
The printed version and path identify the environment and files used by this session.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import tempfile
import shutil

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import kumo.engine as kumo
from kumoapi.model_plan import RunMode
from kumoapi.train import TrainingTableSpec
from kumoapi.typing import Stype

if not torch.cuda.is_available():
    raise RuntimeError("Attach this notebook to Databricks GPU compute.")

output_parent = Path(OUTPUT_PARENT).expanduser().resolve()
output_parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="money-muling-", dir=output_parent))
data_dir = run_dir / "data"
data_dir.mkdir()
kumo.configure(scratch_dir=str(run_dir / "scratch"))
torch.manual_seed(SEED)
print(f"Engine {version('kumo-engine')} | results: {run_dir}")


<br>

## Generate synthetic example data

Run the next cell to create historical accounts, transfers, sender/receiver postings, and future reports, plus a separate network reserved for batch scoring. You can skim the generation code; the Kumo workflow starts at **Build the graph**.

All transfer history precedes the common cutoff. Reports fall in the following 30 days and supply labels only; they stay outside the feature graph. Account target values in the graph are constant placeholders.


In [ ]:
def make_report_network(n_accounts, n_transfers, prefix, seed):
    local_rng = np.random.default_rng(seed)
    ids = np.array([f"synthetic-{prefix}-account-{i:04d}" for i in range(n_accounts)])
    propensity = local_rng.beta(1.2, 5.0, n_accounts)
    activity_weights = (0.5 + 8 * propensity) / (0.5 + 8 * propensity).sum()
    account_rows = pd.DataFrame({
        "account_id": ids,
        "account_age_days": local_rng.integers(HISTORY_DAYS + 1, 2400, n_accounts),
        "account_type": local_rng.choice(["checking", "savings"], n_accounts, p=[0.8, 0.2]),
        "synthetic_region": local_rng.choice(["region_a", "region_b", "region_c"], n_accounts),
        "mule_label": np.zeros(n_accounts, dtype=np.int64),
    })
    senders = local_rng.choice(n_accounts, n_transfers, p=activity_weights)
    receivers = local_rng.choice(n_accounts, n_transfers, p=activity_weights)
    receivers = np.where(receivers == senders, (receivers + 1) % n_accounts, receivers)
    times = cutoff - pd.to_timedelta(local_rng.integers(1, HISTORY_DAYS * 86400, n_transfers), unit="s")
    transfer_rows = pd.DataFrame({
        "transaction_id": [f"synthetic-{prefix}-transfer-{i:05d}" for i in range(n_transfers)],
        "timestamp": times, "amount": np.round(local_rng.lognormal(4.2, 1.0, n_transfers), 2),
        "channel": local_rng.choice(["online", "mobile", "branch"], n_transfers, p=[0.45, 0.5, 0.05]),
    })
    result = {"accounts": account_rows, "transactions": transfer_rows}
    for name, key, positions in (("inputs", "input_id", senders), ("outputs", "output_id", receivers)):
        result[name] = pd.DataFrame({
            key: [f"synthetic-{prefix}-{name}-{i:05d}" for i in range(n_transfers)],
            "transaction_id": transfer_rows.transaction_id,
            "account_id": ids[positions], "timestamp": times,
        })
    flagged = local_rng.random(n_accounts) < REPORT_BASE_RATE + REPORT_SIGNAL * propensity
    report_rows = pd.DataFrame({
        "report_id": [f"synthetic-{prefix}-report-{i:04d}" for i in range(flagged.sum())],
        "account_id": ids[flagged],
        "timestamp": cutoff + pd.to_timedelta(
            local_rng.integers(1, HORIZON_DAYS * 86400 + 1, flagged.sum()), unit="s"
        ),
    })
    return result, report_rows

cutoff = pd.Timestamp(CUTOFF)
historical_frames, reports = make_report_network(N_ACCOUNTS, N_TRANSFERS, "history", SEED)
new_frames, _ = make_report_network(N_NEW_ACCOUNTS, N_NEW_TRANSFERS, "new", SEED + 1)
accounts, transactions = historical_frames["accounts"], historical_frames["transactions"]
inputs, outputs = historical_frames["inputs"], historical_frames["outputs"]
account_ids = accounts.account_id.to_numpy()
frames = {**historical_frames, "reports": reports}
rng = np.random.default_rng(SEED + 2)
display(pd.DataFrame({"historical_rows": {name: len(frame) for name, frame in historical_frames.items()},
                      "reserved_rows": {name: len(frame) for name, frame in new_frames.items()}}))


Check historical and reserved keys, relationships, and time boundaries before constructing the graph.


In [ ]:
primary_keys = {"accounts": "account_id", "transactions": "transaction_id",
                "inputs": "input_id", "outputs": "output_id", "reports": "report_id"}
for collection in (historical_frames, new_frames):
    for name, frame in collection.items():
        assert frame[primary_keys[name]].notna().all() and frame[primary_keys[name]].is_unique
    for name in ("inputs", "outputs"):
        postings = collection[name]
        assert postings.account_id.isin(collection["accounts"].account_id).all()
        assert postings.transaction_id.is_unique
        pd.testing.assert_frame_equal(postings[["transaction_id", "timestamp"]],
                                      collection["transactions"][["transaction_id", "timestamp"]])
    assert collection["inputs"].account_id.ne(collection["outputs"].account_id).all()
    assert collection["transactions"].timestamp.lt(cutoff).all()
    assert np.isfinite(collection["transactions"].amount).all() and collection["transactions"].amount.gt(0).all()
    assert collection["accounts"].account_age_days.gt(HISTORY_DAYS).all()
    assert collection["accounts"].mule_label.eq(0).all()
for name, frame in new_frames.items():
    assert set(frame[primary_keys[name]]).isdisjoint(historical_frames[name][primary_keys[name]])
    assert frame.dtypes.equals(historical_frames[name].dtypes)
assert reports.report_id.is_unique and reports.account_id.isin(accounts.account_id).all()
assert reports.timestamp.gt(cutoff).all()
assert reports.timestamp.le(cutoff + pd.Timedelta(days=HORIZON_DAYS)).all()
print("Keys, historical features, reserved IDs, and future label windows are valid.")


<br>

## Build the graph

Save the tables as local Parquet and define the four sender/receiver relationships with [Engine tables and graphs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs).
The graph includes historical features and a zero-valued target placeholder; future report outcomes remain separate.
Keep the construction helper for the reload step.


In [ ]:
for name, frame in frames.items():
    (data_dir / name).mkdir()
    frame.to_parquet(data_dir / name / "history.parquet", index=False)

def build_graph(data_dir):
    connector = kumo.FileConnector(root_dir=str(data_dir))
    tables = {"accounts": kumo.Table.from_source_table(
        connector["accounts"],
        ["account_id", "account_age_days", "account_type", "synthetic_region", "mule_label"],
        primary_key="account_id",
    )}
    for name, key in (("transactions", "transaction_id"), ("inputs", "input_id"), ("outputs", "output_id")):
        table = kumo.Table.from_source_table(connector[name], primary_key=key, time_column="timestamp")
        if name != "transactions":
            table = table.with_column("account_id", stype=Stype.ID).with_column("transaction_id", stype=Stype.ID)
        tables[name] = table
    relationships = [
        kumo.Edge(src_table=posting, fkey=key, dst_table=destination)
        for posting in ("inputs", "outputs")
        for key, destination in (("account_id", "accounts"), ("transaction_id", "transactions"))
    ]
    graph = kumo.Graph(tables=tables, edges=relationships)
    graph.validate()
    return connector, graph

files, graph = build_graph(data_dir)


The [graph visualization](https://github.com/NVIDIA/kumo-engine/blob/main/packages/kumo-engine/README.md#visualize-table-relationships) below is a preview unavailable in this public release.
Keep this cell commented until graph visualization is available in the installed package.


In [ ]:
# Requires Kumo Engine 1.1
# graph.visualize(show_cols=True)


Materialize a snapshot of the validated historical graph for training and prediction.


In [ ]:
snapshot = graph.snapshot()


<br>

## Define the prediction question

Use a static classification query for this frozen cohort: does an account receive any report in the 30 days after the cutoff?
The comparison defines the task, while the custom training table below supplies the true labels.
This split compares accounts at one cutoff; it does not measure performance across future time periods.


In [ ]:
QUERY = "PREDICT accounts.mule_label = 1 FOR EACH accounts.account_id"
pq = kumo.PredictiveQuery(graph, QUERY)
pq.validate()
print(QUERY)
print("Task:", pq.get_task_type())


<br>

## Separate training and holdout accounts

Partition accounts into training, validation, internal test, and final holdout groups, using the configured fractions.
Historical links may cross partitions, but final holdout labels never enter fitting or model selection.
Scoring requests contain account IDs only.


In [ ]:
labels = pd.DataFrame({"ENTITY1": account_ids, "TARGET1": np.isin(account_ids, reports.account_id)})
account_order = rng.permutation(N_ACCOUNTS)
split_boundaries = np.rint(np.cumsum(SPLIT_FRACTIONS) * N_ACCOUNTS).astype(int)
split = np.empty(N_ACCOUNTS, dtype=np.int64)
for split_id, positions in enumerate(np.split(account_order, split_boundaries[:-1])):
    split[positions] = split_id
labels["SPLIT"] = split
trainval = labels.loc[labels.SPLIT != 3].copy()
heldout = labels.loc[labels.SPLIT == 3].copy()
requests = heldout[["ENTITY1"]].copy()
assert set(trainval.ENTITY1).isdisjoint(heldout.ENTITY1)
assert requests.ENTITY1.is_unique and "TARGET1" not in requests
assert labels.groupby("SPLIT").TARGET1.nunique().eq(2).all(), (
    "Each split needs both classes; increase N_ACCOUNTS or adjust the simulation seed."
)
split_summary = labels.groupby("SPLIT").TARGET1.agg(accounts="size", report_rate="mean")
split_summary.index = ["training", "validation", "internal test", "final holdout"]
display(split_summary)


Create the [custom training table](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training) with explicit labels and splits: 0 for training, 1 for validation, and 2 for internal testing.
Save the final holdout IDs separately for prediction.


In [ ]:
trainval.to_parquet(data_dir / "trainval.parquet", index=False)
requests.to_parquet(data_dir / "requests.parquet", index=False)
training_spec = TrainingTableSpec(split_col="SPLIT")
training = kumo.TrainingTable.from_source_table(
    files["trainval"], training_spec, pq=pq, snapshot=snapshot
)
print(f"Fit population: {len(trainval)} accounts; final holdout: {len(heldout)} accounts.")


<br>

## Fit a comparison baseline

Fit logistic regression on account age and incoming/outgoing transfer aggregates.
Its scaler and estimator use only the training accounts, so both models can be compared on the same final holdout.


In [ ]:
features = accounts.set_index("account_id")[["account_age_days"]].astype(float)
for direction, posting in (("incoming", outputs), ("outgoing", inputs)):
    amounts = posting.merge(transactions[["transaction_id", "amount"]], on="transaction_id", validate="one_to_one")
    aggregates = amounts.groupby("account_id").amount.agg(["count", "sum", "mean"])
    aggregates.columns = [f"{direction}_{name}" for name in aggregates.columns]
    features = features.join(aggregates).fillna(0)
training_accounts = labels.loc[labels.SPLIT == 0]
baseline = make_pipeline(StandardScaler(), LogisticRegression(random_state=SEED, max_iter=1000))
baseline.fit(features.loc[training_accounts.ENTITY1], training_accounts.TARGET1)
print(f"Baseline fitted on {len(training_accounts)} training accounts.")


<br>

## Configure the graph model

Use Engine's suggested FAST plan for this classification task.
Disable refitting so the saved model and internal-test explanations describe the same experiment.


In [ ]:
plan = pq.suggest_model_plan(snapshot, train_table_spec=training_spec, run_mode=RunMode.FAST)
plan.training_job.manual_seed = SEED
plan.training_job.refit_trainval = False
plan.training_job.refit_full = False
plan.training_job.disable_explain = False
print("Training uses the suggested FAST plan.")


<br>

## Train the model

Fit the model on the labeled training table and historical snapshot.
Training saves a model bundle for the reload step.


In [ ]:
trainer = kumo.Trainer(plan)
training_result = trainer.fit(training, snapshot=snapshot)
bundle_dir = Path(training_result.bundle_dir)
print("Saved model:", bundle_dir)


<br>

## Evaluate on holdout accounts

Prepare the final holdout as an ID-only [prediction population](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions). Check results by returned account ID so both models are evaluated on the same accounts.


In [ ]:
def align_predictions(frame, expected_ids):
    assert {"ENTITY", "True_PROB", "False_PROB"} <= set(frame.columns)
    assert frame.ENTITY.is_unique and set(frame.ENTITY) == set(expected_ids)
    aligned = frame.set_index("ENTITY").loc[list(expected_ids)].reset_index()
    probabilities = aligned[["False_PROB", "True_PROB"]].to_numpy()
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-6)
    return aligned

population = kumo.PredictionTable.from_source_table(
    files["requests"], pq=pq, snapshot=snapshot, plan=trainer.training_table_generation_plan
)


Score every holdout account, check the returned IDs and probabilities, and preview the batch.


In [ ]:
scored = trainer.predict(population, snapshot=snapshot, binary_classification_threshold=THRESHOLD)
gnn_predictions = align_predictions(scored.frame, heldout.ENTITY1)
display(gnn_predictions.head())


<br>

## Compare holdout rankings

AUROC and average precision measure how well each model ranks reported accounts above unreported accounts; average precision also depends on report prevalence.
These synthetic-data metrics are **demonstration only**.
The small graph model need not outperform the baseline.


In [ ]:
def quality(actual, probability):
    return {"AUROC": roc_auc_score(actual, probability),
            "Average precision": average_precision_score(actual, probability)}

baseline_probability = baseline.predict_proba(features.loc[heldout.ENTITY1])[:, list(baseline.classes_).index(True)]
metrics = pd.DataFrame({
    "Logistic baseline": quality(heldout.TARGET1, baseline_probability),
    "Engine GNN": quality(heldout.TARGET1, gnn_predictions.True_PROB),
}).T
print(f"Final holdout: {len(heldout)} accounts; report rate: {heldout.TARGET1.mean():.1%}.")
display(metrics)


<br>

## Batch predictions

Append the reserved accounts and their historical transfers/postings to the graph's source directories. They were absent from training and share the same feature cutoff; this demonstrates newly ingested accounts, not performance across a later time period. True report outcomes are never supplied for these accounts.

In [ ]:
for name, frame in new_frames.items():
    frame.to_parquet(data_dir / name / "new.parquet", index=False)
scoring_requests = new_frames["accounts"][["account_id"]].rename(columns={"account_id": "ENTITY1"})
assert set(scoring_requests.columns) == {"ENTITY1"}
assert set(scoring_requests.ENTITY1).isdisjoint(labels.ENTITY1)
scoring_requests.to_parquet(data_dir / "scoring_requests.parquet", index=False)
scoring_snapshot = graph.snapshot()
assert scoring_snapshot.id != snapshot.id
scoring_population = kumo.PredictionTable.from_source_table(
    files["scoring_requests"], pq=pq, snapshot=scoring_snapshot,
    plan=trainer.training_table_generation_plan,
)
accepted_population = pd.read_parquet(scoring_population.table_path)
assert accepted_population.ENTITY1.is_unique
assert set(accepted_population.ENTITY1) == set(scoring_requests.ENTITY1)
assert not any(column.startswith("TARGET") for column in accepted_population)
display(accepted_population.head())

Score the newly added accounts with the fitted model and refreshed snapshot.

In [ ]:
batch_result = trainer.predict(
    scoring_population, snapshot=scoring_snapshot, binary_classification_threshold=THRESHOLD
)

<br>

### Inspect and save account scores

Expect one row per new account. `report_probability` is the simulated-report probability; `review_flag` applies the illustrative threshold. Verify returned IDs, preview the results, and save the full batch as Parquet.


In [ ]:
batch_scores = align_predictions(batch_result.frame, scoring_requests.ENTITY1)
predictions = pd.DataFrame({
    "account_id": scoring_requests.ENTITY1.to_numpy(),
    "report_probability": batch_scores.True_PROB.to_numpy(),
})
predictions["review_flag"] = predictions.report_probability > THRESHOLD
prediction_path = run_dir / "engine_predictions.parquet"
predictions.to_parquet(prediction_path, index=False)
display(predictions.head())
print(f"Saved {len(predictions)} newly added account scores to {prediction_path}.")


<br>

## Reload and score again

The bundle stores the fitted model; retain the feature files, scoring IDs, and compatible package environment separately.
Rebuild the graph snapshot, [load the saved model](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models), and check that it reproduces the account scores without fitting again.


In [ ]:
reloaded_files, reloaded_graph = build_graph(data_dir)
reloaded_snapshot = reloaded_graph.snapshot()
reloaded_query = kumo.PredictiveQuery(reloaded_graph, QUERY)
reloaded_query.validate()
reloaded_trainer = kumo.Trainer.load(str(bundle_dir))
reloaded_population = kumo.PredictionTable.from_source_table(
    reloaded_files["scoring_requests"], pq=reloaded_query, snapshot=reloaded_snapshot,
    plan=reloaded_trainer.training_table_generation_plan,
)
reloaded_result = reloaded_trainer.predict(
    reloaded_population, snapshot=reloaded_snapshot, binary_classification_threshold=THRESHOLD
)
reloaded_predictions = align_predictions(reloaded_result.frame, scoring_requests.ENTITY1)
np.testing.assert_allclose(reloaded_predictions.True_PROB, predictions.report_probability, rtol=1e-5, atol=1e-6)
print(f"Reloaded model reproduced all {len(reloaded_predictions)} account scores.")


<br>

## Inspect training explanations

[Training explanations](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/model-explainability) describe internal-test examples, not the final scoring batch.
Resolve the selected experiment and inspect its highest column sensitivity scores; these are not causal effects.


In [ ]:
training_run = bundle_dir.parent
selection_path, = training_run.rglob("best_trainer.json")
selection = json.loads(selection_path.read_text())
selected_experiment = selection_path.parent / Path(selection["best_trainer"]).name
gradient_path = selected_experiment / "explain_test/scores.csv"
cohort_path = selected_experiment / "column_analysis/test/column_analysis.json"
subgraph_paths = sorted((selected_experiment / "subgraph_collector/test/subgraph_json").glob("*.json"))
assert gradient_path.is_file() and cohort_path.is_file() and subgraph_paths, "Inspect training logs if explanations are missing."
gradient = pd.read_csv(gradient_path)
assert np.isfinite(gradient.score).all()
display(gradient[["table", "column", "score"]].sort_values("score", ascending=False).head(10))


Preview feature-value cohorts with their population shares and average predictions and targets.
Keep the graph hop alongside each column, since the same column can appear at multiple hops.


In [ ]:
cohort_payload = json.loads(cohort_path.read_text())
cohort_rows = []
for entry in cohort_payload["cohorts"]:
    for group, share, predicted, observed in zip(
        entry["cohorts"], entry["populations"], entry["predictions"], entry["targets"], strict=True
    ):
        cohort_rows.append({"table": entry["table_name"], "column": entry["col_name"], "hop": entry["hop"],
                            "group": group, "population_share": share,
                            "mean_prediction": predicted, "mean_target": observed})
cohort_summary = pd.DataFrame(cohort_rows)
display(cohort_summary.head())


Inspect the size of one sampled explanation graph to see how many connected records it contains.


In [ ]:
sampled_graph = json.loads(subgraph_paths[0].read_text())["subgraph_store"]
print(f"Sampled context: {len(sampled_graph['nodes'])} nodes and {len(sampled_graph['edges'])} edges.")


<br>

## Keep or remove this run's files

`CLEANUP=False` keeps the inputs, predictions, model bundle, and explanation files for this notebook session.
These driver-local files are temporary and can disappear when compute is replaced or stopped.
Set `CLEANUP=True` to remove only the unique directory created by this notebook.


In [ ]:
if CLEANUP:
    shutil.rmtree(run_dir)
print("Removed this run's files." if CLEANUP else f"Results retained in {run_dir}.")
